# StarDist, Otsu và Watershed — thực nghiệm tái lập

Notebook dùng chung `run_experiment.py` với CLI: chọn tham số trên 20 ảnh validation, lưu cấu hình, rồi đánh giá 20 ảnh test. Không train/fine-tune và không chuẩn hóa NPY lần hai.

Để xem kết quả đã bàn giao, giữ `RUN_EXPERIMENT=False`. Để chạy mới, chọn thư mục `OUTPUT` chưa có kết quả rồi bật `RUN_EXPERIMENT=True`. Đọc `HUONG_DAN_CHAY_STARDIST.md` và `BAO_CAO_KY_THUAT.md`.

**Chỉ số:** matching accuracy = TP/(TP+FP+FN), tổng hợp micro; không phải COCO AP. Kết quả cũ với nhãn “AP” đã được thay bằng các bảng xuất có định nghĩa rõ ràng.


In [ ]:
from pathlib import Path
import sys
import subprocess

candidates = [Path.cwd(), *Path.cwd().parents,
              Path('/content/BTL_XLA_N3_2026'),
              Path('/kaggle/working/BTL_XLA_N3_2026')]
REPO = next((p for p in candidates if (p / 'run_experiment.py').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Đưa cả repo vào máy chạy và sửa REPO tới thư mục có run_experiment.py.')
DATASET = REPO / 'StarDist_Dataset'
OUTPUT = REPO / 'results/stardist_pretrained'
# Kaggle input chỉ đọc: đổi OUTPUT thành Path('/kaggle/working/results/run_01').
print('Repo:', REPO)
print('Output:', OUTPUT)


In [ ]:
INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', str(REPO / 'requirements-stardist.txt')], check=True)


## Chạy mô hình

Lần đầu cần Internet để tải trọng số. Mỗi phương pháp được warm-up trên validation trước khi đo thời gian. Mã gọi bằng subprocess mới để cấu hình Keras trước khi import TensorFlow. Không dùng kết quả test để chọn lại ngưỡng.


In [ ]:
RUN_EXPERIMENT = False
INCLUDE_BASELINES = True
TUNE_STARDIST_ON_VALIDATION = True
if RUN_EXPERIMENT:
    command = [sys.executable, str(REPO / 'run_experiment.py'),
               '--dataset', str(DATASET), '--output', str(OUTPUT)]
    if not INCLUDE_BASELINES:
        command.append('--stardist-only')
    if not TUNE_STARDIST_ON_VALIDATION:
        command.append('--default-thresholds')
    subprocess.run(command, check=True, cwd=REPO)


In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown, Image

if not (OUTPUT / 'run_metadata.json').is_file():
    raise FileNotFoundError('Chưa có lần chạy hoàn tất. Chọn OUTPUT mới, bật RUN_EXPERIMENT và chạy ô trước.')
metadata = json.loads((OUTPUT / 'run_metadata.json').read_text())
assert metadata['status'] == 'complete'
config = json.loads((OUTPUT / 'selected_config.json').read_text())
print('Tham số đã chốt trên validation:', config['methods'])
display(Markdown((OUTPUT / 'KET_QUA.md').read_text()))
summary = pd.read_csv(OUTPUT / 'metrics_summary.csv')
display(summary[(summary['split'] == 'test') & (summary['iou_threshold'] == 0.5)])


In [ ]:
for filename in ['segmentation_comparison.png', 'matching_accuracy_iou.png',
                 'stardist_probability.png', 'watershed_steps.png']:
    figure = OUTPUT / 'figures' / filename
    if figure.is_file():
        display(Image(filename=str(figure)))
print('ID minh họa:', json.loads((OUTPUT / 'figures/sample_ids.json').read_text()))


In [ ]:
import numpy as np
import tifffile

predictions = pd.read_csv(OUTPUT / 'predictions_manifest.csv')
row = predictions.query("method == 'stardist' and split == 'test'").iloc[0]
labels = tifffile.imread(OUTPUT / row.prediction_path)
print('Source ID:', row.source_id)
print('Mask:', labels.shape, labels.dtype, 'Số nhân:', np.count_nonzero(np.unique(labels)))
display(pd.read_csv(OUTPUT / 'metrics_per_image.csv').head())


## Giới hạn kết luận

`2D_versatile_fluo` đã học một phần DSB2018 theo tài liệu StarDist. Chưa đối chiếu được danh sách huấn luyện pretrained với test cục bộ, nên số liệu chỉ mô tả so sánh trong tập ảnh đã chọn. Ảnh minh họa được chọn theo mật độ nhân, không theo điểm dự đoán. Đọc `BAO_CAO_KY_THUAT.md` để đưa phần phương pháp và giới hạn này vào báo cáo nhóm.
